In [8]:
import pandas as pd
from src.data.clean_retail_data import (
    flag_order_anomalies,
    filter_valid_transactions,
    remove_duplicates,
    rename_column,
)
from src.data.standardization_validation_data import (
    clean_text_base,
    create_description_lookup,
    standardize_descriptions,
    validate_date_range,
    validate_null_rates,
    validate_schema,
)

# Load data
df = pd.read_csv("data/raw/(RAW)online_retail_II.csv")

# Run functional wrangling steps
df = rename_column(df)
df = remove_duplicates(df)
df_clean, df_returns = filter_valid_transactions(df)

# Calculate your boundary parameters explicitly
q1 = df_clean["quantity"].quantile(0.25)
q3 = df_clean["quantity"].quantile(0.75)
iqr = q3 - q1
qnt_upper_bound = q3 + 1.5 * iqr

#Flag and Standardize for Non return data
df_final = flag_order_anomalies(df_clean, qnt_upper_bound)
df_final = clean_text_base(df)
lookup_table_final = create_description_lookup(df)
df_final = standardize_descriptions(df, lookup_table_final)

#Standardization for Reutrn data 
df_returns = clean_text_base(df_returns)
lookup_table_return = create_description_lookup(df_returns)
df_returns = standardize_descriptions(df_returns, lookup_table_return)

# 4. Run automated check rules (will fail loudly if errors are found)
df_final = validate_schema(df)
df_final = validate_null_rates(df)
df_final = validate_date_range(df)

#Validation for Return Data 
df_returns = validate_schema(df_returns)
df_returns = validate_null_rates(df_returns)
df_returns = validate_date_range(df_returns)

# 5. Look at the clean results
df_returns.head()

✅ Schema Validation Passed Successfully!
 Null Rate Threshold Checks Passed!
x Date-Range Controls Passed!
✅ Schema Validation Passed Successfully!
 Null Rate Threshold Checks Passed!
x Date-Range Controls Passed!


,invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,16321.0,Australia
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,16321.0,Australia
180,C489449,21895,UNKNOWN ITEM,-4,2009-12-01 10:33:00,4.25,16321.0,Australia
181,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,16321.0,Australia
182,C489449,22083,PAPER CHAIN KIT RETROSPOT,-12,2009-12-01 10:33:00,2.95,16321.0,Australia
